In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
from dotenv import load_dotenv
import os

DOTENV_PATH = r"C:\Users\Admin\hipython\.env"
load_dotenv(dotenv_path=DOTENV_PATH, override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

In [5]:
loader = PyPDFLoader("simple_rag_pdf_app/data/Samsung_Card_Manual_Korean_1.3.pdf")

In [6]:
pages = loader.load()  # List[Document] 형태로 반환

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.split_documents(pages)

In [16]:
embeddings = OpenAIEmbeddings()

In [20]:
vectordb = FAISS.from_documents(docs, embeddings)

In [22]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [25]:
prompt = ChatPromptTemplate.from_template("""
너는 삼성전자 메모리카드 매뉴얼 전문 어시스턴트이다.
반드시 아래 참고 문서를 근거로만 답변하라.
참고 문서에 없는 내용은 추측하지 말고, 문서에서 확인되지 않는다고 답하라.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하라.
""")

In [28]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [29]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [32]:
query = "이 유틸리티는 동시에 몇 개의 메모리카드나 UFD를 인식할 수 있나?"
answer = rag_chain.invoke(query)

print(answer)

이 유틸리티는 동시에 최대 8개의 메모리 카드나 UFD를 인식할 수 있습니다.
